### Stage 2 – Simplified
Interstitial nodes removed; only true intersections and dead-ends remain.
Edge geometries are preserved via `LineString` attributes.
Nodes are shown in **orange**.

In [8]:
import time
import ducknx as dx
import pandas as pd
from lonboard import viz, ScatterplotLayer, PathLayer, PolygonLayer, Map

dx.settings.pbf_file_path = "berlin-latest.osm.pbf"
BBOX = (13.38, 52.51, 13.42, 52.53)

# --- Stage 1: unsimplified graph ---
t0 = time.perf_counter()
G_raw = dx.graph_from_bbox(BBOX, network_type="drive", simplify=False)
t_raw = time.perf_counter() - t0

# --- Stage 2: simplify ---
t0 = time.perf_counter()
G_simple = dx.simplify_graph(G_raw)
t_simplify = time.perf_counter() - t0

# --- Stage 3: project then consolidate intersections ---
G_proj = dx.project_graph(G_simple)
t0 = time.perf_counter()
G_consolidated = dx.consolidate_intersections(G_proj, tolerance=15, rebuild_graph=True)
t_consolidate = time.perf_counter() - t0

# --- Summary table ---
summary = pd.DataFrame({
    "Stage": ["1 – Raw (unsimplified)", "2 – Simplified", "3 – Consolidated"],
    "Nodes": [len(G_raw), len(G_simple), len(G_consolidated)],
    "Edges": [len(G_raw.edges), len(G_simple.edges), len(G_consolidated.edges)],
    "Time (s)": [f"{t_raw:.3f}", f"{t_simplify:.3f}", f"{t_consolidate:.3f}"],
}).set_index("Stage")

print("Simplification pipeline summary (central Berlin, drive network)")
print(f"  Node reduction:  {len(G_raw):,} → {len(G_simple):,} → {len(G_consolidated):,}")
print(f"  Edge reduction:  {len(G_raw.edges):,} → {len(G_simple.edges):,} → {len(G_consolidated.edges):,}")
summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Simplification pipeline summary (central Berlin, drive network)
  Node reduction:  4,534 → 529 → 373
  Edge reduction:  7,304 → 1,210 → 931


,Nodes,Edges,Time (s)
Stage,,,
1 – Raw (unsimplified),4534,7304,3.995
2 – Simplified,529,1210,0.100
3 – Consolidated,373,931,0.084


In [2]:
# Optimized path
dx.settings.use_duckdb_optimized = True
t0 = time.perf_counter()
G_opt = dx.graph_from_bbox(BBOX, network_type="drive")
t_opt = time.perf_counter() - t0
print(f"Optimized: {len(G_opt):,} nodes, {len(G_opt.edges):,} edges in {t_opt:.3f}s")

# Legacy path
dx.settings.use_duckdb_optimized = False
t0 = time.perf_counter()
G_leg = dx.graph_from_bbox(BBOX, network_type="drive")
t_leg = time.perf_counter() - t0
print(f"Legacy:    {len(G_leg):,} nodes, {len(G_leg.edges):,} edges in {t_leg:.3f}s")

# Comparison
nodes_match = set(G_opt.nodes) == set(G_leg.nodes)
edges_match = len(G_opt.edges) == len(G_leg.edges)
speedup = t_leg / t_opt if t_opt > 0 else 0

comparison = pd.DataFrame({
    "Metric": ["Nodes", "Edges", "Time (s)", "Speedup"],
    "Optimized": [len(G_opt), len(G_opt.edges), f"{t_opt:.3f}", f"{speedup:.2f}x"],
    "Legacy": [len(G_leg), len(G_leg.edges), f"{t_leg:.3f}", ""],
    "Match": [nodes_match, edges_match, "", ""],
}).set_index("Metric")
comparison

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Optimized: 474 nodes, 1,115 edges in 4.077s


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Legacy:    474 nodes, 1,115 edges in 4.105s


,Optimized,Legacy,Match
Metric,,,
Nodes,474,474,True
Edges,1115,1115,True
Time (s),4.077,4.105,
Speedup,1.01x,,


In [3]:
tags = {"building": True}

# Optimized path
dx.settings.use_duckdb_optimized = True
t0 = time.perf_counter()
gdf_opt = dx.features_from_bbox(BBOX, tags=tags)
t_feat_opt = time.perf_counter() - t0
print(f"Optimized: {len(gdf_opt):,} features in {t_feat_opt:.3f}s")
print(f"Geometry types: {dict(gdf_opt.geom_type.value_counts())}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Optimized: 3,394 features in 4.063s
Geometry types: {'Polygon': np.int64(3390), 'Point': np.int64(4)}


In [4]:
gdf_opt

geometry  \
element  id                                                               
node     11264643253                           POINT (13.3973 52.52222)   
         12052263014                          POINT (13.40963 52.52658)   
         12094257580                           POINT (13.39712 52.5289)   
         12532228528                          POINT (13.38381 52.51519)   
relation 3619         POLYGON ((13.39803 52.51945, 13.39912 52.51986...   
...                                                                 ...   
way      1387195651   POLYGON ((13.41226 52.52671, 13.41228 52.52671...   
         1410558398   POLYGON ((13.38844 52.52788, 13.38835 52.52786...   
         1411458709   POLYGON ((13.39894 52.51616, 13.39889 52.51615...   
         1414332366   POLYGON ((13.40609 52.5128, 13.40612 52.51278,...   
         1414332367   POLYGON ((13.40679 52.51311, 13.40682 52.51308...   

                          access         amenity     building  fee  \
element  id                                                          
node     11264643253         yes         toilets          yes   no   
         12052263014         NaN             NaN     bungalow  NaN   
         12094257580         NaN             NaN  outbuilding  NaN   
         12532228528         NaN  security_booth   guardhouse  NaN   
relation 3619                NaN             NaN       museum  yes   
...                          ...             ...          ...  ...   
way      1387195651          NaN             NaN          yes  NaN   
         1410558398          NaN             NaN  residential  NaN   
         1411458709   permissive             NaN      terrace  NaN   
         1414332366          NaN             NaN          yes  NaN   
         1414332367          NaN             NaN          yes  NaN   

                              operator toilets:disposal toilets:position  \
element  id                                                                
node     11264643253      Finizio GmbH       dry_toilet    seated;urinal   
         12052263014  Stromnetz Berlin              NaN              NaN   
         12094257580  Stromnetz Berlin              NaN              NaN   
         12532228528               NaN              NaN              NaN   
relation 3619                      NaN              NaN              NaN   
...                                ...              ...              ...   
way      1387195651                NaN              NaN              NaN   
         1410558398                NaN              NaN              NaN   
         1411458709                NaN              NaN              NaN   
         1414332366                NaN              NaN              NaN   
         1414332367                NaN              NaN              NaN   

                     unisex wheelchair  ... wheelchair:url contact:website:de  \
element  id                             ...                                     
node     11264643253    yes        yes  ...            NaN                NaN   
         12052263014    NaN        NaN  ...            NaN                NaN   
         12094257580    NaN        NaN  ...            NaN                NaN   
         12532228528    NaN        NaN  ...            NaN                NaN   
relation 3619           NaN        yes  ...            NaN                NaN   
...                     ...        ...  ...            ...                ...   
way      1387195651     NaN        NaN  ...            NaN                NaN   
         1410558398     NaN        NaN  ...            NaN                NaN   
         1411458709     NaN        NaN  ...            NaN                NaN   
         1414332366     NaN        NaN  ...            NaN                NaN   
         1414332367     NaN        NaN  ...            NaN                NaN   

                     emergency:phone name:cy name:ga name:gd short_name:pl  \
element  id                                            

## 4. Visualize Building Features with lonboard

In [9]:
from lonboard import SolidPolygonLayer

# Separate polygons and other geometries (keep only geometry for lonboard)
polygons = gdf_opt[gdf_opt.geom_type.isin(["Polygon", "MultiPolygon"])][["geometry"]].copy()
lines = gdf_opt[gdf_opt.geom_type.isin(["LineString", "MultiLineString"])][["geometry"]].copy()
points = gdf_opt[gdf_opt.geom_type == "Point"][["geometry"]].copy()

layers = []

if not polygons.empty:
    poly_layer = SolidPolygonLayer.from_geopandas(
        polygons.reset_index(),
        get_fill_color=[255, 140, 0, 160],
        get_line_color=[80, 80, 80, 200],
    )
    layers.append(poly_layer)

if not lines.empty:
    line_layer = PathLayer.from_geopandas(
        lines.reset_index(),
        width_min_pixels=1,
        get_color=[0, 180, 120, 200],
    )
    layers.append(line_layer)

if not points.empty:
    point_layer = ScatterplotLayer.from_geopandas(
        points.reset_index(),
        get_radius=5,
        radius_min_pixels=2,
        get_fill_color=[100, 100, 255, 200],
    )
    layers.append(point_layer)

print(f"Polygons: {len(polygons):,}, Lines: {len(lines):,}, Points: {len(points):,}")
features_map = Map(layers=layers)
features_map

Polygons: 3,390, Lines: 0, Points: 4


## 5. Combined View: Network + Buildings

In [7]:
combined_layers = []

# Buildings (polygons only)
if not polygons.empty:
    bldg_layer = SolidPolygonLayer.from_geopandas(
        polygons.reset_index(),
        get_fill_color=[220, 180, 100, 120],
        get_line_color=[160, 120, 60, 150],
    )
    combined_layers.append(bldg_layer)

# Road network edges
road_layer = PathLayer.from_geopandas(
    edges_gdf[["geometry"]].reset_index(),
    width_min_pixels=2,
    get_color=[50, 120, 220, 230],
)
combined_layers.append(road_layer)

# Road network nodes
node_layer = ScatterplotLayer.from_geopandas(
    nodes_gdf[["geometry"]].reset_index(),
    get_radius=6,
    radius_min_pixels=1,
    radius_max_pixels=4,
    get_fill_color=[240, 60, 40, 200],
)
combined_layers.append(node_layer)

combined_map = Map(layers=combined_layers)
combined_map

## 6. Sample Data

In [ ]:
gdf_opt.head(10)

## 7. Graph Simplification & Intersection Consolidation

Build an **unsimplified** graph, then run `simplify_graph` and
`consolidate_intersections` to show how the topology is progressively
cleaned up.  Three lonboard maps compare the stages side-by-side.